In [1]:
using Pkg
using Metal

cd(@__DIR__)
Pkg.activate("../")
ParamFile = "../config/testparam.csv"  # maybe GeoPoints and planet1D should be fusioned

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include("../src/batchFiles/batchGPU.jl")


include("../src/commonBatchs.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

include("../src/flexOPT.jl")

using .commonBatchs, .planet1D, .GeoPoints, .flexOPT

  Activating project at `~/Documents/Github/flexOPT`


devs = Metal.devices() = Metal.MTL.MTLDeviceInstance[Metal.MTL.MTLDeviceInstance (object of type AGXG13XDevice)]
→ Using Metal backend (1 device(s))
Selected backend type: MetalBackend


input parameters

In [2]:
#famousEquationType="2DacousticTime"
#Δ = (1.0,1.0,1.0)
#famousEquationType="1DsismoFreqHomo" # GT95
famousEquationType="1DsismoTime" #GT98
#famousEquationType="3DsismoTimeIso" 
Δ = (1.0,1.0)

# orders: -1 -> indicator function, 0 -> box car, >=1 -> B-spline

orderBtime=1
orderBspace=1

pointsInSpace=3
pointsInTime=3

# the order of errors to be controlled
supplementaryOrder=2

# new parameters for interpolated Taylor expansion μ for field

fieldItpl = (ptsSpace = 1,ptsTime = 1,offsetSpace=1,offsetTime=1,YorderBspace=-1,YorderBtime=1) #offsetSpace and offsetTime ∈ z 
# μ points should be distributed from y_min+offset*Δy to y_max-offset*Δy offset can be negative too


# new parameters for interpolated Taylor expansion μᶜ for material
materItpl = (ptsSpace = 1,ptsTime = 1,offsetSpace=1,offsetTime=1,YorderBspace=-1,YorderBtime=1)



(ptsSpace = 1, ptsTime = 1, offsetSpace = 1, offsetTime = 1, YorderBspace = -1, YorderBtime = 1)

In [3]:
concreteParametersForOPTConstruction = @strdict famousEquationType Δ orderBtime orderBspace pointsInSpace pointsInTime supplementaryOrder fieldItpl materItpl


Dict{String, Any} with 9 entries:
  "fieldItpl"          => (ptsSpace = 1, ptsTime = 1, offsetSpace = 1, offsetTi…
  "Δ"                  => (1.0, 1.0)
  "supplementaryOrder" => 2
  "materItpl"          => (ptsSpace = 1, ptsTime = 1, offsetSpace = 1, offsetTi…
  "orderBspace"        => 1
  "orderBtime"         => 1
  "famousEquationType" => "1DsismoTime"
  "pointsInSpace"      => 3
  "pointsInTime"       => 3

In [4]:
 Ajiννᶜ, AjiννᶜU,Ulocal=makeOPTsemiSymbolic(concreteParametersForOPTConstruction)

(vars, iVars) = ((ρ(x), μ(x)), 1)
(vars, iVars) = ((ρ(x), μ(x)), 2)
(vars, iVars) = ((ρ(x), μ(x)), 1)
(vars, iVars) = ((ρ(x), μ(x)), 2)
pointsIndices = availablePointsConfigurations[iConfigGeometry] = SVector{2, Int64}[[1, 1] [1, 2] [1, 3]; [2, 1] [2, 2] [2, 3]; [3, 1] [3, 2] [3, 3]]
middleLinearν = centrePointConfigurations[iConfigGeometry] = 5
μPoints = availableμPoints[iConfigGeometry] = SVector{2, Float64}[[2.0, 2.0];;]
μᶜPoints = availableμᶜPoints[iConfigGeometry] = SVector{2, Float64}[[2.0, 2.0];;]
μaxes = availableμaxes[iConfigGeometry] = ([2.0], [2.0])
μᶜaxes = availableμᶜaxes[iConfigGeometry] = ([2.0], [2.0])
size(μPoints) = (1, 1)
pointν = pointsIndices[middleLinearν] = [2, 2]
(typeof(μPoints), μPoints[1], typeof(pointsIndices)) = (Matrix{SVector{2, Float64}}, [2.0, 2.0], Matrix{SVector{2, Int64}})
nTotalSmallα = sum((length(bigα[iExpr, iField]) for iField = 1:NtypeofFields, iExpr = 1:NtypeofExpr)) = 3
all_sizes = collect.(size.(coefWYYKK)) = [[5, 5, 1, 1, 1], [5, 5, 1, 1, 1]

BoundsError: BoundsError: attempt to access Dict{String, @NamedTuple{lhs::@NamedTuple{Ajiννᶜ::Array{Symbolics.Num, 4}, Ulocal::Matrix{Symbolics.Num}, varM::Matrix{Any}, CartesianDependencies::Matrix{Int64}}, rhs::@NamedTuple{Γjiννᶜ::Array{Symbolics.Num, 4}, Flocal::Matrix{Symbolics.Num}, varF::Matrix{Any}, CartesianDependencies::Matrix{Int64}}, nodes::Vector{Any}, centresIndices::Vector{Int64}, numbersOfTheSystem::@NamedTuple{numbersOfTheSystemL::@NamedTuple{timeMarching::Bool, NtypeofExpr::Int64, NtypeofMaterialVariables::Int64, NtypeofFields::Int64, nCoordinates::Int64, Ndimension::Val{2}, pointsUsed::Vector{Int64}, pointsμUsed::Vector{Int64}, offsetsμUsed::Vector{Float64}, pointsμᶜUsed::Vector{Int64}, offsetsμᶜUsed::Vector{Float64}}, numbersOfTheSystemR::@NamedTuple{timeMarching::Bool, NtypeofExpr::Int64, NtypeofMaterialVariables::Int64, NtypeofFields::Int64, nCoordinates::Int64, Ndimension::Val{2}, pointsUsed::Vector{Int64}, pointsμUsed::Vector{Int64}, offsetsμUsed::Vector{Float64}

In [5]:
 Ajiννᶜ

"recette" => (lhs = (Ajiννᶜ = Symbolics.Num[-0.041494258μ₁ - 0.041494258μ₂ + 3.259629e-9μ₃ + 0.056888063ρ₁ + 0.04986822ρ₂ - 0.023767779ρ₃; 0.041494254μ₁ + 0.082988515μ₂ + 0.041494254μ₃ + 0.04986823ρ₁ + 0.7342866ρ₂ + 0.04986823ρ₃; … ; 0.041494254μ₁ + 0.08298852μ₂ + 0.041494254μ₃ + 0.04986823ρ₁ + 0.7342866ρ₂ + 0.04986823ρ₃; 3.7252903e-9μ₁ - 0.04149426μ₂ - 0.041494258μ₃ - 0.023767779ρ₁ + 0.04986822ρ₂ + 0.05688806ρ₃;;;;], Ulocal = Symbolics.Num[u₁; u₂; … ; u₈; u₉;;], varM = Any[ρ₁ ρ₂ … ρ₂ ρ₃; μ₁ μ₂ … μ₂ μ₃], CartesianDependencies = [1 1; 0 0]), rhs = (Γjiννᶜ = Symbolics.Num[0.0068870923f0; 0.06921429f0; … ; 0.06921428f0; 0.0068870923f0;;;;], Flocal = Symbolics.Num[f₁; f₂; … ; f₈; f₉;;], varF = Any[f₁ f₂ … f₈ f₉], CartesianDependencies = [1; 1;;]), nodes = Any[SVector{2, Int64}[[1, 1] [1, 2] [1, 3]; [2, 1] [2, 2] [2, 3]; [3, 1] [3, 2] [3, 3]]], centresIndices = [5], numbersOfTheSystem = (numbersOfTheSystemL = (timeMarching = true, NtypeofExpr = 1, NtypeofMaterialVariables = 2, NtypeofFields